# Running AlgoDisco Directly with the Python API

This tutorial demonstrates how to run AlgoDisco directly from Python code without YAML configuration files. You will create configs, LLM providers, evaluators, and execute the search loop programmatically.


## Table of Contents
1. [Prerequisites](#1-prerequisites)
2. [Creating Config Objects](#2-creating-config-objects)
3. [Creating LLM Providers](#3-creating-llm-providers)
4. [Creating Evaluators](#4-creating-evaluators)
5. [Running Search](#5-running-search)
6. [Inspecting Results](#6-inspecting-results)
7. [Complete Example](#7-complete-example)

## 1. Prerequisites

In [ ]:
# Set up API Key
import os
os.environ["OPENAI_API_KEY"] = "your-api-key"  # Replace with your actual API key

### Template Program

The template program provides the algorithm skeleton, including function signatures and partial implementation. The LLM will complete the missing parts.

In [ ]:
# Template program - algorithm skeleton with function signature and partial implementation
template_code = '''
from typing import List

def sort_list(lst: List[int]) -> List[int]:
    """
    Sort a list of integers in ascending order.

    Args:
        lst: A list of integers to sort

    Returns:
        A new list with integers sorted in ascending order
    """
    # TODO: Implement the sorting algorithm
    pass
'''
print("Template Program:")
print(template_code)

### Task Description

The task description instructs the LLM on what problem to solve. Be specific about requirements, constraints, and examples.

In [ ]:
# Task description - instructs the LLM on what problem to solve
task_description = '''
You are an expert programmer. Your task is to implement a sorting algorithm.

Requirements:
1. The function should take a list of integers
2. Return a new list sorted in ascending order
3. Do not modify the original list
4. Handle edge cases: empty list, single element

Example:
Input: [3, 1, 4, 1, 5]
Output: [1, 1, 3, 4, 5]

Important:
- Do not use built-in sorted() or list.sort()
- Implement your own sorting algorithm (e.g., bubble sort, quick sort)
'''
print("Task Description:")
print(task_description)

## 2. Creating Config Objects

The `FunSearchConfig` object holds all search parameters. Here's a comprehensive breakdown of available options:

In [ ]:
from algodisco.methods.funsearch.config import FunSearchConfig

# Create FunSearchConfig with detailed parameters
config = FunSearchConfig(
    # === Template and Task ===
    template_program=template_code,  # Template program code (string)
    task_description=task_description,  # Task description (string)
    language="python",  # Programming language
    
    # === Parallelism Configuration ===
    num_samplers=2,  # Number of parallel sampling processes
    num_evaluators=2,  # Number of parallel evaluation processes
    
    # === Prompt Configuration ===
    examples_per_prompt=2,  # Number of examples shown in each prompt
    samples_per_prompt=2,  # Number of samples generated per prompt
    
    # === Search Limits ===
    max_samples=100,  # Maximum total samples to generate
    
    # === LLM Configuration ===
    llm_max_tokens=1024,  # Maximum tokens the LLM can generate
    llm_timeout_seconds=60,  # Timeout for LLM calls in seconds
    
    # === Database (Island Model) Configuration ===
    # The database uses an island model for genetic diversity
    db_num_islands=5,  # Number of islands in the population
    db_reset_period=7200,  # Reset period in seconds (0 to disable)
)

print("Config created successfully!")
print(f"Template program length: {len(config.template_program)} characters")
print(f"Max samples: {config.max_samples}")
print(f"Number of islands: {config.db_num_islands}")

### Configuration Options Reference

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `template_program` | str | Required | Template code string |
| `task_description` | str | Required | Task instruction string |
| `language` | str | Required | Programming language (python, java, etc.) |
| `num_samplers` | int | 1 | Parallel sampling processes |
| `num_evaluators` | int | 1 | Parallel evaluation processes |
| `examples_per_prompt` | int | 1 | Examples shown in prompts |
| `samples_per_prompt` | int | 1 | Samples generated per prompt |
| `max_samples` | int | 100 | Maximum total samples |
| `llm_max_tokens` | int | 1024 | LLM max output tokens |
| `llm_timeout_seconds` | int | 60 | LLM call timeout |
| `db_num_islands` | int | 5 | Number of islands |
| `db_reset_period` | int | 7200 | Island reset period (seconds) |

## 3. Creating LLM Providers

AlgoDisco supports multiple LLM backends. Here are the available options.


In [ ]:
from algodisco.providers.llm.openai_api import OpenAIAPI

# Option 1: OpenAI API
llm = OpenAIAPI(
    model="gpt-4o-mini",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url="https://api.openai.com/v1",
)

# Option 2: Claude API (Anthropic)
# from algodisco.providers.llm.claude_api import ClaudeAPI
# llm = ClaudeAPI(
#     model="claude-3-5-sonnet-latest",
#     api_key=os.environ["ANTHROPIC_API_KEY"],
# )

# Option 3: vLLM launcher
# from algodisco.providers.llm.vllm_server import VLLMServer
# llm = VLLMServer(
#     model_path="Qwen/Qwen2.5-0.5B-Instruct",
#     port=8000,
#     gpus=0,
#     launch_vllm_in_init=False,
# )

# Option 4: SGLang launcher
# from algodisco.providers.llm.sglang_server import SGLangServer
# llm = SGLangServer(
#     model_path="Qwen/Qwen2.5-0.5B-Instruct",
#     port=30000,
#     gpus=0,
#     launch_sglang_in_init=False,
# )

print("LLM provider created successfully!")
print(type(llm).__name__)


## 4. Creating Evaluators

Evaluators assess the quality of generated algorithms. You must implement a class that inherits from `Evaluator` and implements the `evaluate_program` method.

In [ ]:
# Create a custom Evaluator for sorting algorithms
import os
import random
import subprocess
import sys
import tempfile
from algodisco.base.evaluator import Evaluator, EvalResult

class SortingEvaluator(Evaluator):
    """Evaluator for sorting algorithm correctness"""

    def __init__(self, num_tests=100):
        self.num_tests = num_tests
        self.test_cases = self._generate_test_cases()

    def _generate_test_cases(self):
        """Generate random test cases with known correct outputs"""
        cases = []
        for _ in range(self.num_tests):
            n = random.randint(0, 20)
            case = random.sample(range(-100, 100), n)
            expected = sorted(case)
            cases.append((case, expected))
        return cases

    def evaluate_program(self, program_str: str) -> EvalResult:
        """
        Evaluate a program by:
        1. Writing it to a temporary file
        2. Executing test cases
        3. Computing a score based on correctness
        """
        temp_path = None
        try:
            with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
                f.write(program_str)
                f.flush()
                temp_path = f.name

            correct = 0
            test_subset = self.test_cases[:20]

            for inputs, expected in test_subset:
                test_code = f"""
import sys
sys.path.insert(0, '.')
exec(open("{temp_path}").read())
result = sort_list({inputs})
print(result == {expected})
"""
                exec_result = subprocess.run(
                    [sys.executable, "-c", test_code],
                    capture_output=True,
                    timeout=5,
                )
                if exec_result.returncode == 0 and "True" in exec_result.stdout.decode():
                    correct += 1

            score = correct / len(test_subset)

            return {
                "score": score,
                "execution_time": 0.0,
                "error_msg": None,
            }

        except subprocess.TimeoutExpired:
            return {
                "score": 0.0,
                "execution_time": 5.0,
                "error_msg": "Timeout",
            }
        except Exception as e:
            return {
                "score": 0.0,
                "execution_time": 0.0,
                "error_msg": str(e)[:200],
            }
        finally:
            if temp_path and os.path.exists(temp_path):
                os.unlink(temp_path)

# Create evaluator instance
evaluator = SortingEvaluator(num_tests=100)
print("Evaluator created successfully!")
print(f"Number of test cases: {evaluator.num_tests}")


### Evaluator Return Format

The `evaluate_program` method must return a dictionary with:
- `score` (float): Performance score between 0.0 and 1.0
- `execution_time` (float): Time taken to execute
- `error_msg` (str or None): Error message if execution failed

## 5. Running Search

Now create the FunSearch instance and run the search loop.

In [ ]:
from algodisco.methods.funsearch.search import FunSearch

# Create FunSearch instance with all components
search = FunSearch(
    config=config,
    llm=llm,
    evaluator=evaluator,
    # logger=logger,  # Optional: add a logger for tracking
)

print("FunSearch instance created successfully!")
print("Calling search.run() to start the search...")

In [ ]:
# Run search (uncomment to execute)
# search.run()

# To change the stopping condition, update the config before running:
# config.max_samples = 50
# search.run()


## 6. Inspecting Results

After the search completes, the evaluated programs live in `search._database`.
The most useful methods in the current implementation are:

- `get_all_algorithms()` to collect every stored `AlgoProto`
- `gather_all_programs_grouped_by_islands()` to inspect diversity by island
- `get_island_stats()` to see how many programs each island holds


In [ ]:
# Get all stored algorithms and inspect the best ones
all_programs = search._database.get_all_algorithms()
ranked_programs = sorted(
    all_programs,
    key=lambda p: p.score if p.score is not None else float("-inf"),
    reverse=True,
)

print(f"Stored programs: {len(ranked_programs)}")

if ranked_programs:
    best_program = ranked_programs[0]
    print(f"Best score: {best_program.score}")
    print(f"Best program preview:\n{best_program.program[:400]}")

    print("\nTop 5 programs:")
    for i, program in enumerate(ranked_programs[:5], 1):
        print(f"Rank {i}: score={program.score:.3f}, id={program.algo_id}")

high_scoring = [p for p in ranked_programs if (p.score or 0.0) >= 0.8]
print(f"High-scoring programs (score >= 0.8): {len(high_scoring)}")


### Result Inspection Patterns

These examples use the database methods that exist in the current FunSearch implementation.


In [ ]:
import json

# Pattern 1: Inspect island-level diversity
island_groups = search._database.gather_all_programs_grouped_by_islands()
for island_id, programs in enumerate(island_groups):
    print(f"Island {island_id}: {len(programs)} programs")

# Pattern 2: Get compact database stats
print("Island stats:", search._database.get_island_stats())
print("Sample count seen by search:", search.current_num_samples())

# Pattern 3: Find programs with recorded errors
error_programs = [
    p for p in search._database.get_all_algorithms() if p.metadata.get("error_msg")
]
print(f"Programs with errors: {len(error_programs)}")

# Pattern 4: Export results to JSON
export_data = [p.to_dict() for p in search._database.get_all_algorithms()]
with open("results.json", "w") as f:
    json.dump(export_data, f, indent=2)
print("Saved results to results.json")


## 7. Complete Example

Putting it all together - a complete, runnable example with all components.

In [ ]:
# Complete Example: Running AlgoDisco with the Python API
#
# This example combines all steps to run a sorting algorithm search.
# Replace the placeholder API key before running.

import os
import subprocess
import sys
import tempfile

# Set API key
os.environ["OPENAI_API_KEY"] = "your-api-key"  # Replace with your actual API key

from algodisco.base.evaluator import Evaluator
from algodisco.methods.funsearch.config import FunSearchConfig
from algodisco.methods.funsearch.search import FunSearch
from algodisco.providers.llm.openai_api import OpenAIAPI

# === 1. Prepare Data ===
template_code = (
    "from typing import List\n\n"
    "def sort_list(lst: List[int]) -> List[int]:\n"
    '    """Sort a list of integers in ascending order."""\n'
    "    pass\n"
)

task_description = (
    "Implement a sorting algorithm. Do not use built-in sorted() or list.sort().\n"
    "Your implementation should handle edge cases like empty lists.\n"
)

# === 2. Create Config ===
config = FunSearchConfig(
    template_program=template_code,
    task_description=task_description,
    language="python",
    num_samplers=2,
    num_evaluators=2,
    samples_per_prompt=2,
    max_samples=10,
    llm_max_tokens=512,
)

# === 3. Create LLM Provider ===
llm = OpenAIAPI(
    model="gpt-4o-mini",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url="https://api.openai.com/v1",
)

# === 4. Create Evaluator ===
class SimpleEvaluator(Evaluator):
    """Minimal evaluator for quick testing."""

    def __init__(self):
        self.test_cases = [
            ([3, 1, 2], [1, 2, 3]),
            ([5, 4, 3, 2, 1], [1, 2, 3, 4, 5]),
            ([], []),
            ([1], [1]),
        ]

    def evaluate_program(self, program_str):
        temp_path = None
        try:
            with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
                f.write(program_str)
                f.flush()
                temp_path = f.name

            passed = 0
            for case, expected in self.test_cases:
                test_code = (
                    f"exec(open('{temp_path}').read()); "
                    f"result = sort_list({case}); "
                    f"print(result == {expected})"
                )
                result = subprocess.run(
                    [sys.executable, "-c", test_code],
                    capture_output=True,
                    timeout=5,
                )
                if result.returncode == 0 and "True" in result.stdout.decode():
                    passed += 1

            return {"score": passed / len(self.test_cases)}
        except Exception as e:
            return {"score": 0.0, "error_msg": str(e)}
        finally:
            if temp_path and os.path.exists(temp_path):
                os.unlink(temp_path)

# === 5. Run Search ===
evaluator = SimpleEvaluator()
search = FunSearch(config=config, llm=llm, evaluator=evaluator)

print("Ready to run search. Uncomment the next lines to execute:")
# search.run()
# ranked = sorted(
#     search._database.get_all_algorithms(),
#     key=lambda p: p.score if p.score is not None else float("-inf"),
#     reverse=True,
# )
# print(f"Best score: {ranked[0].score}")


## Summary

Running AlgoDisco with the Python API involves these steps:

1. **Prepare `template_program` and `task_description`** - Pass them as strings
2. **Create `FunSearchConfig`** - Configure search parameters
3. **Create an LLM provider** - Choose from OpenAI, Claude, vLLM, or SGLang
4. **Create an evaluator** - Extend the Evaluator base class and implement `evaluate_program`
5. **Create a `FunSearch` instance** - Pass config, llm, and evaluator
6. **Call `search.run()`** - Execute the search loop
7. **Inspect results** - Review scores and generated programs

### When to Use Python API vs YAML

| Approach | Use Case |
|----------|----------|
| **Python API** | Prototyping, custom workflows, integration with other code |
| **YAML Config** | Reproducible experiments, configuration management, deployment |
